# Historical Pricing get_data_async with Asyncio.TaskGroup Performance (30 RICs)

Importing requires libraries

In [1]:
import os
import time
import asyncio
from IPython.display import Markdown, display
import lseg.data as ld
from lseg.data import session
from lseg.data._errors import LDError
from lseg.data.content import historical_pricing
from lseg.data.content.historical_pricing import Adjustments, Intervals
from dotenv import load_dotenv

Loading Platform session (Data Platform) credential from `.env` file.

In [2]:
# Load environment variables from .env file
load_dotenv(dotenv_path='.env')
# Retrieve Platform Session credentials from environment variables
app_key = os.getenv('LSEG_API_KEY')
username = os.getenv('LSEG_MACHINE_ID')
password = os.getenv('LSEG_PASSWORD')

Define required variables.

In [3]:
# -- Top 30 S&P 500 companies ----------------------------------------------------
INSTRUMENTS = {
    "NVDA.O":  "NVIDIA",
    "AAPL.O":  "Apple",
    "MSFT.O":  "Microsoft",
    "AMZN.O":  "Amazon",
    "GOOG.O":  "Alphabet",
    "AVGO.O":  "Broadcom",
    "META.O":  "Meta",
    "ORCL.N":  "Oracle",
    "IBM.N":   "IBM",
    "PLTR.O":  "Palantir",
    "NFLX.O":  "Netflix",
    "TSLA.O":  "Tesla",
    "CRM.N":   "Salesforce",
    "AMD.O":   "AMD",
    "INTC.O":  "Intel",
    "ARM.O":   "Arm Holdings",
    "TXN.O": "Texas Instruments",
    "CSCO.O":  "Cisco Systems",
    "WMT.O":   "Walmart",
    "LLY.N":   "Eli Lilly and Company",
    "JPM.N":   "JPMorgan Chase & Co.",
    "XOM.N":   "Exxon Mobil Corporation",
    "V.N":     "Visa Inc.",
    "JNJ.N":   "Johnson & Johnson",
    "MU.O":    "Micron Technology, Inc.",
    "MA.N":    "Mastercard Incorporated",
    "COST.O":  "Costco Wholesale Corporation",
    "CVX.N":   "Chevron Corporation",
    "BAC.N":   "Bank of America Corporation",
    "CAT.N":   "Caterpillar Inc.",
}

# ── Date range ─────────────────────────────────────────────────────────────────
START = "2026-02-01T00:00:00Z"
#END   = "2026-06-30T23:59:59Z"
END = "2026-02-28T23:59:59Z"

# ── Event correction filters ───────────────────────────────────────────────────
EVENT_ADJUSTMENTS = [Adjustments.EXCHANGE_CORRECTION,
                    Adjustments.MANUAL_CORRECTION,
                    Adjustments.CCH,
                    Adjustments.CRE,
                    Adjustments.RPO,
                    Adjustments.RTS]

# ── Field lists ─────────────────────────────────────────────────────────────────
INTERDAY_FIELDS = ["BID", "ASK", "OPEN_PRC", "HIGH_1", "LOW_1", "TRDPRC_1", "NUM_MOVES", "TRNOVR_UNS"]
MAX_CONCURRENT_REQUESTS = 10  # Maximum number of concurrent requests to the API

Intitalize and open the session.

In [4]:
# Create a platform session with provided credentials for authentication
ld_session = session.platform.Definition(
         app_key=app_key,
         grant=session.platform.GrantPassword(
             username=username,
             password=password
         ),
         signon_control=True
).get_session()

# Set this session as the default for all subsequent Data Library calls
session.set_default(ld_session)

# Open the connection to the LSEG Data Platform
ld_session.open()

<OpenState.Opened: 'Opened'>

Define `display_response` for the `Asyncio TaskGroup`.

In [5]:
def display_response(data, format="df"):
    """Display results from asyncio.TaskGroup tasks.

    TaskGroup raises an ExceptionGroup if any task fails, so by the time
    results reach this function every item is a successful API response.
    """
    for idx, api_response in enumerate(data, start=1):
        closure = getattr(api_response, "closure", f"result-{idx}")
        print(f"\n[{idx}] Response received for: {closure}")

        if api_response.is_success:
            if format == "df":
                display(api_response.data.df)
            elif format == "json":
                display(api_response.data.raw) 
        else:
            # HTTP-level failure (4xx / 5xx from the platform).
            print(f"Request failed - HTTP status: {api_response.http_status}")

Creating a helper method to send Historical Pricing Summaries Definition with `asyncio.Semaphore` and group requests in to a list of coroutine.

In [6]:
# Convert the company dictionary to a list once.
# Example item: ("NVDA.O", "NVIDIA")
list_of_rics_companies = list(INSTRUMENTS.items())

semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)  # Number of simultaneous tasks to run

async def fetch_with_throttle(ric, company):
    """Request historical summaries for one RIC with semaphore throttling."""
    # Semaphore enforces max concurrent requests across all created coroutines.
    async with semaphore:
        return await historical_pricing.summaries.Definition(
            universe=ric,
            fields=INTERDAY_FIELDS,
            start=START,
            end=END,
            adjustments=EVENT_ADJUSTMENTS,
            interval=Intervals.DAILY,
        ).get_data_async(closure=company)

# Build request coroutines; TaskGroup in the next cell schedules them concurrently.
interday_summary_tasks = [
    fetch_with_throttle(ric, company)
    for ric, company in list_of_rics_companies
]

Requests 30 Instruments asynchronously with time counter. 

In [7]:
display(Markdown("**Start the wall-clock timer...**"))
start_time = time.perf_counter()

# Run one daily-summary request per instrument at the same time.
# TaskGroup waits for all tasks to finish before moving on.
try:
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(coro) for coro in interday_summary_tasks]

    # Collect completed responses in the same order as 'instruments'.
    historical_data = [task.result() for task in tasks]
except* LDError as errors:
    for error in errors.exceptions:
        print(error)
finally:
    elapsed = time.perf_counter() - start_time
    elapsed_string = f"**Sending Historical-Pricing Interday Summaries (Daily) for 30 RICs from {START} to {END} with concurrency limit of {MAX_CONCURRENT_REQUESTS} using TaskGroup ends in {elapsed:0.2f} seconds.**"
    display(Markdown(elapsed_string))

**Start the wall-clock timer...**

**Sending Historical-Pricing Interday Summaries (Daily) for 30 RICs from 2026-02-01T00:00:00Z to 2026-02-28T23:59:59Z with concurrency limit of 10 using TaskGroup ends in 12.02 seconds.**

The timing shown above measures only the request/response phase and does not include data display time.

In this notebook, output can be rendered in JSON form (`response.data.raw`). If you choose to use DataFrame output (`response.data.df`) instead, expect additional processing time for converting JSON results into a DataFrame.

In [8]:
try:
    display(Markdown("**30 Companies Historical Pricing Interday Summaries (Daily) via Iteration**"))
    display_response(historical_data, format="json")
except Exception as errors:
    print(f"An exception occurred: {errors}")

**30 Companies Historical Pricing Interday Summaries (Daily) via Iteration**


[1] Response received for: NVIDIA


{'universe': {'ric': 'NVDA.O'},
 'interval': 'P1D',
 'summaryTimestampLabel': 'endPeriod',
 'adjustments': ['exchangeCorrection',
  'manualCorrection',
  'CCH',
  'CRE',
  'RTS',
  'RPO'],
 'defaultPricingField': 'TRDPRC_1',
 'headers': [{'name': 'DATE', 'type': 'string'},
  {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'HIGH_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'LOW_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'OPEN_PRC', 'type': 'number', 'decimalChar': '.'},
  {'name': 'BID', 'type': 'number', 'decimalChar': '.'},
  {'name': 'ASK', 'type': 'number', 'decimalChar': '.'},
  {'name': 'TRNOVR_UNS', 'type': 'number', 'decimalChar': '.'},
  {'name': 'NUM_MOVES', 'type': 'number', 'decimalChar': '.'}],
 'data': [['2026-02-27',
   177.19,
   182.59,
   176.38,
   181.25,
   177.14,
   177.15,
   55882286062,
   3823737],
  ['2026-02-26',
   184.89,
   194.29,
   184.315,
   194.27,
   184.87,
   184.89,
   67579404866,
   4877577],
  ['2026


[2] Response received for: Apple


{'universe': {'ric': 'AAPL.O'},
 'interval': 'P1D',
 'summaryTimestampLabel': 'endPeriod',
 'adjustments': ['exchangeCorrection',
  'manualCorrection',
  'CCH',
  'CRE',
  'RTS',
  'RPO'],
 'defaultPricingField': 'TRDPRC_1',
 'headers': [{'name': 'DATE', 'type': 'string'},
  {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'HIGH_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'LOW_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'OPEN_PRC', 'type': 'number', 'decimalChar': '.'},
  {'name': 'BID', 'type': 'number', 'decimalChar': '.'},
  {'name': 'ASK', 'type': 'number', 'decimalChar': '.'},
  {'name': 'TRNOVR_UNS', 'type': 'number', 'decimalChar': '.'},
  {'name': 'NUM_MOVES', 'type': 'number', 'decimalChar': '.'}],
 'data': [['2026-02-27',
   264.18,
   272.81,
   262.89,
   272.81,
   264.17,
   264.21,
   19221957495,
   768081],
  ['2026-02-26',
   272.95,
   276.11,
   270.795,
   274.945,
   272.99,
   273.01,
   8846778363,
   607281],
  ['2026-0


[3] Response received for: Microsoft


... (51 outputs left out)


[29] Response received for: Bank of America Corporation


{'universe': {'ric': 'BAC.N'},
 'interval': 'P1D',
 'summaryTimestampLabel': 'endPeriod',
 'adjustments': ['exchangeCorrection',
  'manualCorrection',
  'CCH',
  'CRE',
  'RTS',
  'RPO'],
 'defaultPricingField': 'TRDPRC_1',
 'headers': [{'name': 'DATE', 'type': 'string'},
  {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'HIGH_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'LOW_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'OPEN_PRC', 'type': 'number', 'decimalChar': '.'},
  {'name': 'BID', 'type': 'number', 'decimalChar': '.'},
  {'name': 'ASK', 'type': 'number', 'decimalChar': '.'},
  {'name': 'TRNOVR_UNS', 'type': 'number', 'decimalChar': '.'},
  {'name': 'NUM_MOVES', 'type': 'number', 'decimalChar': '.'}],
 'data': [['2026-02-27',
   49.83,
   51.41,
   49.32,
   51.34,
   49.8,
   49.81,
   1506147994,
   61761],
  ['2026-02-26', 52.3, 52.67, 51.75, 51.88, 52.29, 52.3, 1451883232, 42276],
  ['2026-02-25', 51.69, 51.76, 50.43, 50.6, 51.7, 51.71


[30] Response received for: Caterpillar Inc.


{'universe': {'ric': 'CAT.N'},
 'interval': 'P1D',
 'summaryTimestampLabel': 'endPeriod',
 'adjustments': ['exchangeCorrection',
  'manualCorrection',
  'CCH',
  'CRE',
  'RTS',
  'RPO'],
 'defaultPricingField': 'TRDPRC_1',
 'headers': [{'name': 'DATE', 'type': 'string'},
  {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'HIGH_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'LOW_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'OPEN_PRC', 'type': 'number', 'decimalChar': '.'},
  {'name': 'BID', 'type': 'number', 'decimalChar': '.'},
  {'name': 'ASK', 'type': 'number', 'decimalChar': '.'},
  {'name': 'TRNOVR_UNS', 'type': 'number', 'decimalChar': '.'},
  {'name': 'NUM_MOVES', 'type': 'number', 'decimalChar': '.'}],
 'data': [['2026-02-27',
   742.83,
   751.49,
   731.29,
   745.22,
   742.84,
   743.14,
   786363021,
   8767],
  ['2026-02-26',
   752.93,
   767.09,
   728.47,
   763.83,
   752.73,
   752.74,
   592145453,
   13544],
  ['2026-02-25',
  

Closing the session.

In [9]:
# Close the session to release resources; in a long-running application, consider keeping the session open and reusing it for subsequent API calls instead.
ld_session.close()
ld.close_session()